# Floating Catchment Area (FCA) Library — Demo

This notebook demonstrates all five FCA methods available in the library:

| Method | Reference | Key Feature |
|--------|-----------|-------------|
| **2SFCA** | Luo & Wang (2003) | Classic two-step ratio approach |
| **E2SFCA** | Luo & Qi (2009) | Concentric distance-decay zones |
| **3SFCA** | Wan et al. (2012) | Competition / selection weights |
| **M2SFCA** | Delamater (2013) | Normalised continuous decay |
| **KD2SFCA** | Dai (2010) | Kernel density decay |

## 1. Setup and Sample Data

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

import floating_catchment as fca
from floating_catchment import decay

print(f"floating-catchment v{fca.__version__}")

In [ ]:
# --- Supply locations (e.g. hospitals / clinics) ---
supply_data = pd.DataFrame({
    "name":     ["Hospital A", "Hospital B", "Clinic C", "Clinic D"],
    "capacity": [50, 30, 20, 15],
    "latitude":  [40.730, 40.750, 40.710, 40.765],
    "longitude": [-73.990, -73.970, -73.950, -73.985],
})
supply_gdf = fca.load_points(supply_data, id_col="name")
supply_gdf

In [ ]:
# --- Demand locations (e.g. population centres / census tracts) ---
demand_data = pd.DataFrame({
    "name":       [f"Pop_{i}" for i in range(1, 13)],
    "population": [5000, 8000, 3000, 12000, 6000, 4500, 7000, 9500, 2000, 11000, 3500, 6500],
    "latitude":   [40.720, 40.735, 40.742, 40.755, 40.728, 40.718, 40.760, 40.748, 40.770, 40.738, 40.715, 40.752],
    "longitude":  [-73.998, -73.985, -73.975, -73.968, -73.960, -73.948, -73.992, -73.958, -73.980, -73.942, -73.995, -73.970],
})
demand_gdf = fca.load_points(demand_data, id_col="name")
demand_gdf.head()

## 2. Build the Cost Matrix

Two options:
- **Euclidean distance** (computed on-the-fly)
- **Pre-computed travel time** (loaded from an ESRI OD Cost Matrix export)

Here we use Euclidean distance on the WGS 84 coordinates.  
For real analysis, project to a local CRS first (distances in metres).

In [ ]:
cost_matrix = fca.euclidean_distance_matrix(supply_gdf, demand_gdf)
print(f"Cost matrix shape: {cost_matrix.shape}  (demand x supply)")
cost_matrix.round(4)

### Alternative: Load a pre-computed travel-time matrix

```python
# ESRI OD Cost Matrix long format (OriginID, DestinationID, Total_Cost)
cost_matrix = fca.load_cost_matrix("od_cost_matrix.csv", fmt="long")

# Wide-format matrix (rows=demand, columns=supply)
cost_matrix = fca.load_cost_matrix("travel_times.csv", fmt="wide")
```

## 3. Distance Decay Functions

The library provides pluggable decay functions.  Let's visualise them.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.linspace(0, 0.05, 200)
d0 = 0.04  # threshold

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(d, decay.binary(d, d0),        label="Binary")
ax.plot(d, decay.linear(d, d0),        label="Linear")
ax.plot(d, decay.gaussian(d, d0),      label="Gaussian (β=1)")
ax.plot(d, decay.epanechnikov(d, d0),  label="Epanechnikov")
ax.plot(d, decay.butterworth(d, d0),   label="Butterworth (n=2)")
ax.plot(d, decay.power(d, d0, alpha=1.5), label="Power (α=1.5)")
ax.axvline(d0, color="grey", linestyle="--", alpha=0.5, label=f"threshold={d0}")
ax.set_xlabel("Distance")
ax.set_ylabel("Weight")
ax.set_title("Distance Decay Functions")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Run All Five FCA Methods

In [ ]:
supply_cap = supply_gdf["capacity"]
demand_pop = demand_gdf["population"]
threshold = 0.04  # in degrees for this demo

results = {}

### 4a. 2SFCA (Luo & Wang 2003)

$$R_j = \frac{S_j}{\sum_{k: d_{kj} \le d_0} D_k}$$
$$A_i = \sum_{j: d_{ij} \le d_0} R_j$$

In [ ]:
results["2SFCA"] = fca.two_sfca(
    supply_cap, demand_pop, cost_matrix,
    threshold=threshold,
)
results["2SFCA"]

### 4b. E2SFCA (Luo & Qi 2009)

Adds concentric zones with step-wise weights:

$$R_j = \frac{S_j}{\sum_r \sum_{k \in D_r} D_k \cdot W_r}$$

In [ ]:
results["E2SFCA"] = fca.e2sfca(
    supply_cap, demand_pop, cost_matrix,
    zones=[0.015, 0.03, 0.04],
    weights=[1.0, 0.68, 0.22],
)
results["E2SFCA"]

### 4c. 3SFCA (Wan et al. 2012)

Adds a selection-probability step to model competition:

$$T_{ij} = \frac{G(d_{ij}) \cdot S_j}{\sum_{j'} G(d_{ij'}) \cdot S_{j'}}$$

In [ ]:
results["3SFCA"] = fca.three_sfca(
    supply_cap, demand_pop, cost_matrix,
    threshold=threshold,
    decay_fn="gaussian",
)
results["3SFCA"]

### 4d. M2SFCA (Delamater 2013)

Normalises decay weights per demand location to sum to 1:

$$\hat{f}_{ij} = \frac{f(d_{ij})}{\sum_{j'} f(d_{ij'})}$$

In [ ]:
results["M2SFCA"] = fca.m2sfca(
    supply_cap, demand_pop, cost_matrix,
    threshold=threshold,
    decay_fn="gaussian",
)
results["M2SFCA"]

### 4e. KD2SFCA (Dai 2010)

Uses a kernel density function (Epanechnikov by default):

$$K(u) = \frac{3}{4}(1 - u^2), \quad u = d/d_0$$

In [ ]:
results["KD2SFCA"] = fca.kd2sfca(
    supply_cap, demand_pop, cost_matrix,
    threshold=threshold,
    kernel="epanechnikov",
)
results["KD2SFCA"]

## 5. Compare Methods Side-by-Side

In [ ]:
comparison = pd.DataFrame(results)
comparison.index.name = "Demand Location"
comparison.round(6)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
comparison.plot(kind="bar", ax=ax)
ax.set_ylabel("Accessibility Index")
ax.set_title("FCA Methods Comparison")
ax.legend(title="Method")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Visualisation

### 6a. Static Map (matplotlib)

In [ ]:
from floating_catchment.viz import plot_accessibility, interactive_map

# Attach scores to the demand GeoDataFrame
viz_gdf = demand_gdf.copy()
viz_gdf["accessibility"] = results["2SFCA"]

fig = plot_accessibility(
    viz_gdf,
    column="accessibility",
    title="2SFCA Accessibility Index",
    supply_gdf=supply_gdf,
)
plt.show()

### 6b. Interactive Map (Folium)

In [ ]:
m = interactive_map(
    viz_gdf,
    column="accessibility",
    supply_gdf=supply_gdf,
    supply_label_col="name",
    demand_label_col="name",
)
m

## 7. Side-by-Side Maps for All Methods

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, (method_name, scores) in enumerate(results.items()):
    gdf = demand_gdf.copy()
    gdf["accessibility"] = scores
    plot_accessibility(
        gdf,
        column="accessibility",
        title=method_name,
        supply_gdf=supply_gdf,
        ax=axes[idx],
        legend=False,
    )

# Hide unused subplot
axes[-1].set_visible(False)

fig.suptitle("Spatial Accessibility — All FCA Methods", fontsize=16)
plt.tight_layout()
plt.show()

## 8. Using a Pre-Computed Travel Time Matrix

If you have an ESRI OD Cost Matrix export, load it like this:

```python
# Long format (3-column: OriginID, DestinationID, Total_Cost)
cost = fca.load_cost_matrix(
    "path/to/od_cost_matrix.csv",
    fmt="long",
    origin_col="OriginID",
    dest_col="DestinationID",
    cost_col="Total_Minutes",
)

# Then use it with any method
scores = fca.two_sfca(supply_cap, demand_pop, cost, threshold=30)
```

The `threshold` in this case would be in the same units as your cost column (e.g. minutes).